# Preprocessing — Détection de fraude bancaire

**Cours MLOps — notebook 2/4.** Transformer les transactions brutes en données d'entraînement,
et produire les **processeurs** (encodeurs, scaler) que l'API réutilise en production.

**Pipeline**

1. Extraction du sous-ensemble `fraud_600k.csv` (versionné avec DVC)
2. Contrôles et nettoyage
3. Feature engineering — identique à l'API
4. Découpage **train / validation / test** (stratifié)
5. Encodage et normalisation **appris sur le train uniquement**
6. Rééquilibrage SMOTE **sur le train uniquement**
7. Sauvegarde des processeurs et des jeux de données
8. Contrôle de cohérence entraînement / production

> **Règle d'or** : tout ce qui « apprend » quelque chose des données (encodeur, scaler, SMOTE)
> est ajusté sur le train, puis seulement *appliqué* à la validation, au test et en production.
> Sinon, l'évaluation est optimiste : c'est une **fuite de données** (*data leakage*).

## 0. Configuration

In [ ]:
import json
import pickle
import sys
import warnings
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

from project_setup import ROOT, DATA_DIR, PREPROCESSING_DIR, env_flag

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

SOURCE_FILE = DATA_DIR / 'fraud_1M.csv'     # jeu complet (analysé dans le notebook 01)
SUBSET_FILE = DATA_DIR / 'fraud_600k.csv'   # données de modélisation, et référence du drift (Evidently)
N_ROWS = 600_000
TEST_SIZE = 0.20                 # part du jeu total réservée au test final
VAL_SIZE = 0.20                  # part du jeu d'entraînement réservée à la validation
RANDOM_STATE = 42
REBUILD_SUBSET = env_flag('REBUILD_SUBSET')   # REBUILD_SUBSET=1 pour régénérer fraud_600k.csv

PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)
print(f"Racine du projet : {ROOT}")
print(f"Processeurs      : {PREPROCESSING_DIR.relative_to(ROOT)}")

## 1. Extraction du sous-ensemble de 600 000 transactions

Le fichier source est trié par date : les 600 000 premières lignes couvrent donc les
**premiers mois de 2019**. `fraud_600k.csv` est versionné avec DVC : s'il est déjà présent
(après `dvc pull`), on le réutilise tel quel, ce qui garantit que tout le monde travaille sur
exactement les mêmes données.

In [ ]:
if SUBSET_FILE.exists() and not REBUILD_SUBSET:
    print(f"{SUBSET_FILE.name} déjà présent : réutilisation (versionné par DVC).")
else:
    if not SOURCE_FILE.exists():
        raise FileNotFoundError(
            f"Ni {SUBSET_FILE.name} ni {SOURCE_FILE.name} ne sont présents : "
            "téléchargez fraudTrain.csv (Kaggle) sous data/raws/fraud_1M.csv, "
            "ou `dvc pull data/` si les données ont déjà été versionnées (README_GITHUB_DVC_MLFLOW.md § 6.2)."
        )
    df_full = pd.read_csv(SOURCE_FILE)
    df_subset = df_full.head(N_ROWS)
    print(f"Jeu complet   : {len(df_full):,} lignes, taux de fraude {df_full['is_fraud'].mean():.3%}")
    print(f"Sous-ensemble : {len(df_subset):,} lignes, taux de fraude {df_subset['is_fraud'].mean():.3%}")
    print(f"Période       : {df_subset['trans_date_trans_time'].min()} → {df_subset['trans_date_trans_time'].max()}")
    df_subset.to_csv(SUBSET_FILE, index=False)
    del df_full, df_subset
    print(f"Écrit : {SUBSET_FILE.relative_to(ROOT)} — à versionner : `dvc add data` puis `dvc push`.")

In [ ]:
df = pd.read_csv(SUBSET_FILE, parse_dates=['trans_date_trans_time', 'dob'])
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Mémoire    : {df.memory_usage(deep=True).sum() / 1024**2:,.0f} Mo")
print(f"Période    : {df['trans_date_trans_time'].min()} → {df['trans_date_trans_time'].max()}")
df.head()

## 2. Contrôles rapides

In [ ]:
print(f"Valeurs manquantes        : {df.isnull().sum().sum():,}")
print(f"trans_num dupliqués       : {df['trans_num'].duplicated().sum():,}")
print(f"Montants <= 0             : {(df['amt'] <= 0).sum():,}")

fraud_dist = df['is_fraud'].value_counts().sort_index()
print(f"\nNon-fraude (0) : {fraud_dist[0]:,} ({fraud_dist[0] / len(df):.2%})")
print(f"Fraude     (1) : {fraud_dist[1]:,} ({fraud_dist[1] / len(df):.2%})")

fig, ax = plt.subplots(figsize=(7, 4))
fraud_dist.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_xticklabels(['Non-fraude', 'Fraude'], rotation=0)
ax.set_title('Distribution des classes', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Nettoyage

In [ ]:
df_clean = df.drop(columns=['Unnamed: 0'], errors='ignore')
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='trans_num')
print(f"Colonne d'index supprimée, doublons retirés : {n_before - len(df_clean):,}")
print(f"Dataset nettoyé : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")

## 4. Feature engineering

Ces transformations doivent rester **strictement identiques** à celles de l'API
(`InferencePreprocessor.preprocess_inference()` dans `backend/src/api.py`). Une divergence,
même minime (une borne de tranche, une définition de période), crée un
***training-serving skew*** : le modèle reçoit en production des données différentes de
celles vues à l'entraînement, sans qu'aucune erreur ne soit levée. La section 11 le vérifie
automatiquement.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Distance orthodromique en km entre le client et le commerçant."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def get_period(hour):
    if 6 <= hour < 12:
        return 'matin'
    if 12 <= hour < 18:
        return 'après-midi'
    if 18 <= hour < 22:
        return 'soirée'
    return 'nuit'


ts = df_clean['trans_date_trans_time']
df_clean['trans_hour'] = ts.dt.hour
df_clean['trans_day'] = ts.dt.day
df_clean['trans_month'] = ts.dt.month
df_clean['trans_year'] = ts.dt.year
df_clean['trans_dayofweek'] = ts.dt.dayofweek
df_clean['is_weekend'] = (df_clean['trans_dayofweek'] >= 5).astype(int)
df_clean['age'] = (ts - df_clean['dob']).dt.days / 365.25
df_clean['distance_km'] = haversine_distance(df_clean['lat'], df_clean['long'],
                                             df_clean['merch_lat'], df_clean['merch_long'])
df_clean['amt_category'] = pd.cut(df_clean['amt'], bins=[0, 50, 100, 200, float('inf')],
                                  labels=['faible', 'moyen', 'élevé', 'très_élevé'])
df_clean['day_period'] = df_clean['trans_hour'].apply(get_period)

new_features = ['trans_hour', 'trans_day', 'trans_month', 'trans_year', 'trans_dayofweek',
                'is_weekend', 'age', 'distance_km', 'amt_category', 'day_period']
print(f"{len(new_features)} features créées : {', '.join(new_features)}")
df_clean[new_features].describe(include='all').T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
df_clean['trans_hour'].plot(kind='hist', bins=24, ax=axes[0, 0], edgecolor='black', title='Heure')
df_clean['trans_dayofweek'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 1], title='Jour (0 = lundi)')
df_clean['age'].plot(kind='hist', bins=30, ax=axes[0, 2], edgecolor='black', title='Âge')
df_clean['distance_km'].plot(kind='hist', bins=30, ax=axes[1, 0], edgecolor='black', title='Distance (km)')
df_clean['amt_category'].value_counts().reindex(['faible', 'moyen', 'élevé', 'très_élevé']).plot(
    kind='bar', ax=axes[1, 1], title='Tranche de montant')
df_clean['day_period'].value_counts().reindex(['matin', 'après-midi', 'soirée', 'nuit']).plot(
    kind='bar', ax=axes[1, 2], title='Période de la journée')
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 5. Sélection des variables

Chaque colonne conservée est déclarée **catégorielle** ou **numérique** : ces deux listes sont
sauvegardées dans `feature_names.pkl` et pilotent le preprocessing de l'API.

In [ ]:
COLS_TO_DROP = ['trans_date_trans_time', 'cc_num', 'merchant', 'first', 'last',
                'street', 'trans_num', 'dob', 'unix_time', 'city']
CATEGORICAL_FEATURES = ['category', 'gender', 'state', 'job', 'amt_category', 'day_period']
NUMERICAL_FEATURES = ['amt', 'zip', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long',
                      'trans_hour', 'trans_day', 'trans_month', 'trans_year', 'trans_dayofweek',
                      'is_weekend', 'age', 'distance_km']
TARGET = 'is_fraud'

df_model = df_clean.drop(columns=COLS_TO_DROP)
X = df_model.drop(columns=TARGET)
y = df_model[TARGET]

undeclared = set(X.columns) ^ (set(CATEGORICAL_FEATURES) | set(NUMERICAL_FEATURES))
assert not undeclared, f"Colonnes non déclarées ou absentes : {undeclared}"
ALL_FEATURES = X.columns.tolist()   # l'ordre des colonnes fait partie du contrat avec l'API

print(f"Features : {len(ALL_FEATURES)} ({len(CATEGORICAL_FEATURES)} catégorielles, {len(NUMERICAL_FEATURES)} numériques)")
print(f"Taux de fraude : {y.mean():.2%}")

## 6. Découpage train / validation / test

| Jeu | Rôle | Part |
|---|---|---|
| **train** | ajuster encodeurs, scaler, SMOTE et modèles | 64 % |
| **validation** | comparer les modèles, régler les hyperparamètres et le seuil de décision | 16 % |
| **test** | mesurer la performance finale, **une seule fois** | 20 % |

Le découpage est **stratifié** : chaque jeu conserve le taux de fraude d'origine (≈ 0,6 %).
Le test est obtenu avec la même graine et la même proportion que dans les versions
précédentes du projet : ses résultats restent comparables.

> **Pour aller plus loin** : un découpage **temporel** (entraîner sur janvier–avril, tester sur
> mai–juin) est plus proche de la production, où l'on prédit toujours le futur. Il donne des
> scores plus faibles, mais plus réalistes.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE, stratify=y_trainval, random_state=RANDOM_STATE)

split_summary = pd.DataFrame({
    'lignes': [len(X_train), len(X_val), len(X_test)],
    'fraudes': [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
}, index=['train', 'validation', 'test'])
split_summary['taux_fraude_%'] = split_summary['fraudes'] / split_summary['lignes'] * 100
split_summary['part_%'] = split_summary['lignes'] / len(X) * 100
split_summary.round(2)

## 7. Encodage des variables catégorielles

Les modèles attendent des nombres : `LabelEncoder` remplace chaque modalité par un entier.
Il est ajusté **sur le train seulement**. Une modalité absente du train (rare métier, petit
État) peut apparaître en validation, en test ou en production : on la remplace alors par la
première classe connue, **exactement comme le fait l'API**.

In [ ]:
def fit_label_encoders(frame, columns):
    return {col: LabelEncoder().fit(frame[col].astype(str)) for col in columns}


def apply_label_encoders(frame, encoders):
    """Encode les colonnes catégorielles ; valeur inconnue -> première classe connue (comme l'API)."""
    frame = frame.copy()
    unseen = {}
    for col, encoder in encoders.items():
        values = frame[col].astype(str)
        known = values.isin(encoder.classes_)
        unseen[col] = int((~known).sum())
        frame[col] = encoder.transform(values.where(known, encoder.classes_[0]))
    return frame, unseen


label_encoders = fit_label_encoders(X_train, CATEGORICAL_FEATURES)
X_train_enc, _ = apply_label_encoders(X_train, label_encoders)
X_val_enc, unseen_val = apply_label_encoders(X_val, label_encoders)
X_test_enc, unseen_test = apply_label_encoders(X_test, label_encoders)

pd.DataFrame({
    'classes apprises (train)': {col: len(enc.classes_) for col, enc in label_encoders.items()},
    'valeurs inconnues (validation)': unseen_val,
    'valeurs inconnues (test)': unseen_test,
})

## 8. Normalisation des variables numériques

`RobustScaler` centre sur la médiane et divise par l'écart interquartile : il résiste aux
montants extrêmes. Il n'est appliqué qu'aux **variables numériques** — normaliser des codes de
catégories n'a pas de sens.

Les modèles d'arbres (Random Forest, XGBoost, LightGBM, CatBoost) n'ont pas besoin de
normalisation ; on la conserve pour la régression logistique du *stacking* (notebook 03) et
parce que l'API applique le scaler à la liste `numerical_features`. Le scaler est ajusté **sur
exactement cette liste** : le contrat entre `scaler.pkl` et `feature_names.pkl` est ainsi garanti.

In [ ]:
scaler = RobustScaler().fit(X_train_enc[NUMERICAL_FEATURES])


def scale(frame):
    frame = frame.copy()
    frame[NUMERICAL_FEATURES] = scaler.transform(frame[NUMERICAL_FEATURES])
    return frame


X_train_scaled = scale(X_train_enc)
X_val_scaled = scale(X_val_enc)
X_test_scaled = scale(X_test_enc)

examples = ['amt', 'city_pop', 'age', 'distance_km']
pd.DataFrame({
    'médiane avant': X_train_enc[examples].median(),
    'IQR avant': X_train_enc[examples].quantile(0.75) - X_train_enc[examples].quantile(0.25),
    'médiane après': X_train_scaled[examples].median(),
    'IQR après': X_train_scaled[examples].quantile(0.75) - X_train_scaled[examples].quantile(0.25),
}).round(3)

## 9. Rééquilibrage des classes (SMOTE) — sur le train uniquement

SMOTE crée des fraudes **synthétiques** par interpolation entre une fraude et ses plus proches
voisines. Il ne s'applique **qu'au train** : la validation et le test doivent refléter la
réalité (≈ 0,6 % de fraudes), sinon les métriques mesurées n'ont plus de sens.

> **Limite connue.** SMOTE interpole aussi les colonnes catégorielles encodées : entre
> `grocery_pos` (code 4) et `travel` (code 13), il peut produire le code 8,6, qui ne correspond à
> aucune catégorie. Les arbres de décision s'en accommodent (ils ne font que des seuils), mais
> c'est une approximation. `SMOTENC` traite correctement les catégories ; il est ici trop
> gourmand en mémoire (près de 500 métiers distincts). L'alternative sans données synthétiques —
> la **pondération des classes** — est comparée dans le notebook 03 ; c'est pourquoi le train
> non rééchantillonné est aussi sauvegardé.

In [ ]:
counts_before = Counter(y_train)
k_neighbors = min(5, counts_before[1] - 1)
smote = SMOTE(k_neighbors=k_neighbors, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
counts_after = Counter(y_train_res)

print(f"Avant SMOTE : {counts_before[0]:,} non-fraudes / {counts_before[1]:,} fraudes (ratio {counts_before[0] / counts_before[1]:.0f}:1)")
print(f"Après SMOTE : {counts_after[0]:,} non-fraudes / {counts_after[1]:,} fraudes")
print(f"Fraudes synthétiques créées : {len(X_train_res) - len(X_train_scaled):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in [(axes[0], counts_before, 'Avant SMOTE (train)'), (axes[1], counts_after, 'Après SMOTE (train)')]:
    ax.bar(['Non-fraude', 'Fraude'], [counts[0], counts[1]], color=['#2ecc71', '#e74c3c'])
    ax.set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Sauvegarde des processeurs et des jeux de données

| Fichier | Contenu | Utilisé par |
|---|---|---|
| `scaler.pkl` | `RobustScaler` ajusté sur `numerical_features` | API, notebook 03 |
| `label_encoders.pkl` | un `LabelEncoder` par colonne catégorielle | API |
| `feature_names.pkl` | listes `categorical_features`, `numerical_features`, `all_features` (ordre) | API |
| `smote_config.pkl` | paramètres du rééchantillonnage | traçabilité |
| `preprocessed_data.pkl` | `X_train`/`y_train` (SMOTE), `X_val`/`y_val`, `X_test`/`y_test`, `X_train_orig`/`y_train_orig` (sans SMOTE) | notebooks 03–04, `jenkins/train_model.py` |
| `preprocessing_summary.json` | volumes et taux de fraude par jeu | traçabilité |

Les noms et formats des fichiers lus par l'API et par Jenkins sont inchangés.

In [ ]:
feature_names = {
    'categorical_features': CATEGORICAL_FEATURES,
    'numerical_features': NUMERICAL_FEATURES,
    'all_features': ALL_FEATURES,
}
smote_config = {
    'applied': True,
    'strategy': 'SMOTE',
    'random_state': RANDOM_STATE,
    'k_neighbors': k_neighbors,
    'samples_created': len(X_train_res) - len(X_train_scaled),
}
datasets = {
    'X_train': X_train_res, 'y_train': y_train_res,
    'X_val': X_val_scaled, 'y_val': y_val,
    'X_test': X_test_scaled, 'y_test': y_test,
    'X_train_orig': X_train_scaled, 'y_train_orig': y_train,
}

for filename, obj in [('scaler.pkl', scaler), ('label_encoders.pkl', label_encoders),
                      ('feature_names.pkl', feature_names), ('smote_config.pkl', smote_config),
                      ('preprocessed_data.pkl', datasets)]:
    with open(PREPROCESSING_DIR / filename, 'wb') as f:
        pickle.dump(obj, f)
    print(f"✓ {filename}")

summary = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'source_file': SUBSET_FILE.name,
    'n_features': len(ALL_FEATURES),
    'splits': {
        name: {'rows': int(len(y_part)), 'frauds': int(y_part.sum()), 'fraud_rate': float(y_part.mean())}
        for name, y_part in [('train_smote', y_train_res), ('train_orig', y_train), ('validation', y_val), ('test', y_test)]
    },
    'smote': smote_config,
}
with open(PREPROCESSING_DIR / 'preprocessing_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("✓ preprocessing_summary.json")

## 11. Contrôle de cohérence entraînement / production

On recharge les processeurs **comme le fait l'API** — en utilisant directement la classe
`InferencePreprocessor` de `backend/src/api.py` — puis on transforme des transactions brutes
du jeu de test. Le résultat doit être identique, à l'arrondi près, à `X_test` calculé dans ce
notebook. Si ce contrôle échoue, le modèle déployé recevrait des données différentes de celles
de l'entraînement.

Cette cellule nécessite les dépendances du backend (`pip install -r backend/requirements.txt`) ;
sinon, elle est ignorée avec un message.

In [ ]:
sys.path.insert(0, str(ROOT / 'backend' / 'src'))
try:
    from api import InferencePreprocessor
except ImportError as exc:
    print(f"Contrôle ignoré : dépendance du backend manquante ({exc}).")
else:
    inference = InferencePreprocessor()   # charge artifacts/preprocessing/*.pkl, comme l'API
    sample_idx = X_test.sample(min(2_000, len(X_test)), random_state=RANDOM_STATE).index
    raw_columns = [c for c in df.columns if c not in ('Unnamed: 0', TARGET)]
    X_api = inference.preprocess_inference(df_clean.loc[sample_idx, raw_columns])

    assert list(X_api.columns) == ALL_FEATURES, "Ordre ou liste des colonnes différents"
    max_gap = np.abs(X_api.to_numpy(dtype=float) - X_test_scaled.loc[sample_idx].to_numpy(dtype=float)).max()
    assert max_gap < 1e-6, f"Écart entraînement/production : {max_gap}"
    print(f"✅ Preprocessing de l'API identique à celui de l'entraînement ({len(sample_idx):,} transactions, écart max {max_gap:.1e})")

## 12. Résumé et versionnement

- Les transformations sont apprises sur le train ; validation et test ne servent qu'à mesurer.
- `feature_names.pkl` et `scaler.pkl` sont cohérents entre eux et avec l'API (contrôle 11).
- Le jeu de validation permet au notebook 03 de choisir modèle, hyperparamètres et seuil
  **sans toucher au test**.

**Versionner les données et les artefacts** (depuis la racine du dépôt), **avant** le
notebook 03 : ses runs MLflow enregistrent les hash DVC de cette version.

```bash
dvc add data artifacts       # crée (première fois) ou met à jour data.dvc et artifacts.dvc
git add data.dvc artifacts.dvc .gitignore
git commit -m "données : préprocessing"
dvc push                     # envoie les fichiers vers le remote DVC (DagsHub), avant git push
git push
```

**Étape suivante** : `03_fraud_detection_modeling.ipynb`.